# SciGuide Pipeline Demo on BEIR SciFact

Этот ноутбук показывает базовый сценарий работы библиотеки `sciguide_pipeline`:

- загрузка датасета `BEIR SciFact`;
- индексация корпуса через `PipelineManager`;
- демонстрация поиска;
- расчёт метрик `ndcg@k`, `recall@k`, `precision@k`, `mrr@k`.

Ноутбук ориентирован на локальный исследовательский запуск. Все секреты лежат в `notebooks/.env`, а все остальные параметры вынесены в константы прямо в ячейки.

## Какие сервисы нужно поднять

Для этого ноутбука нужны только:

- `qdrant`
- `neo4j`

В директории `notebooks` уже добавлен файл `docker-compose.notebook.yml`. Запускать его лучше из корня репозитория:

```bash
docker compose -f notebooks/docker-compose.notebook.yml up -d
```

Для ноутбука не требуются `db`, `redis`, `flyway` и `sciguide-backend`.

## Подготовка окружения

1. Скопируй `notebooks/.env.example` в `notebooks/.env`.
2. Заполни `OPENROUTER_API_KEY`.
3. При первом запуске установи notebook-зависимости, если их ещё нет в kernel:

```python
%pip install beir pandas tqdm ipywidgets
```

Если выполнял `%pip install` в текущем kernel только что, перезапусти kernel перед запуском следующих ячеек. Без этого `ipywidgets` может не подхватиться, и notebook progress bar не отрисуется.


In [1]:
from __future__ import annotations

import math
import os
import sys
from pathlib import Path
from pprint import pprint

from tqdm.notebook import tqdm as notebook_tqdm
from tqdm.std import tqdm as std_tqdm


def resolve_tqdm_backend():
    try:
        from ipywidgets import IntProgress  # noqa: F401
    except ImportError:
        return std_tqdm, "text"

    return notebook_tqdm, "notebook"


tqdm, tqdm_backend = resolve_tqdm_backend()
if tqdm_backend == "text":
    print(
        "ipywidgets is unavailable in the current kernel, so tqdm will use a text progress bar. "
        "If you just installed ipywidgets, restart the kernel and rerun the notebook."
    )

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = Path.cwd() / "notebooks"

PROJECT_ROOT = NOTEBOOK_DIR.parent
PACKAGES_DIR = PROJECT_ROOT / "src" / "packages"
DATA_DIR = NOTEBOOK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

if str(PACKAGES_DIR) not in sys.path:
    sys.path.insert(0, str(PACKAGES_DIR))

from sciguide_pipeline import PipelineManager, SourceDocument


ipywidgets is unavailable in the current kernel, so tqdm will use a text progress bar. If you just installed ipywidgets, restart the kernel and rerun the notebook.


In [2]:
def load_env_file(env_path: Path) -> None:
    if not env_path.exists():
        raise FileNotFoundError(
            f"Create {env_path} from notebooks/.env.example before running the notebook."
        )

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


load_env_file(NOTEBOOK_DIR / ".env")

OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]

# Secrets come from notebooks/.env. Everything else stays editable here.
QDRANT_URL = "http://localhost:6333"
QDRANT_COLLECTION_NAME = "scifact_notebook"
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "password"
NEO4J_DATABASE = "neo4j"
GRAPH_NAMESPACE = "scifact_notebook"

OPENROUTER_MODEL_NAME = "nvidia/nemotron-3-super-120b-a12b:free"
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
MODEL_CACHE_DIR = DATA_DIR / "hf_cache"

CHUNK_SIZE = 900
CHUNK_OVERLAP = 120
SEARCH_LIMIT = 10
SEARCH_CANDIDATE_LIMIT = 30

# Full BEIR SciFact is more expensive because chunk indexing uses an LLM.
# Start with a subset and then scale up when you are ready.
MAX_CORPUS_DOCUMENTS = 300
MAX_QUERIES = 100
K_VALUES = [5, 10]

In [3]:
def load_scifact_dataset(dataset_root: Path):
    try:
        from beir import util
        from beir.datasets.data_loader import GenericDataLoader
    except ImportError as error:
        raise ImportError(
            "Install notebook dependencies first: %pip install beir pandas"
        ) from error

    dataset_root.mkdir(parents=True, exist_ok=True)
    dataset_dir = dataset_root / "scifact"
    if not dataset_dir.exists():
        url = (
            "https://public.ukp.informatik.tu-darmstadt.de/"
            "thakur/BEIR/datasets/scifact.zip"
        )
        util.download_and_unzip(url, str(dataset_root))

    corpus, queries, qrels = GenericDataLoader(str(dataset_dir)).load(
        split="test"
    )
    return corpus, queries, qrels


corpus, queries, qrels = load_scifact_dataset(DATA_DIR)
len(corpus), len(queries), len(qrels)


/Users/timur/Desktop/FU/KR2/.venv/lib/python3.13/site-packages/beir/util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
100%|██████████| 5183/5183 [00:00<00:00, 213222.28it/s]


(5183, 300, 300)

In [4]:
def build_source_documents(corpus: dict, max_documents: int):
    selected_doc_ids = list(corpus.keys())[:max_documents]
    documents = []

    for doc_id in selected_doc_ids:
        raw_document = corpus[doc_id]
        title = raw_document.get("title", "").strip()
        body = raw_document.get("text", "").strip()
        content = "\n\n".join(part for part in [title, body] if part)
        source_name = f"scifact::{doc_id}"

        documents.append(
            SourceDocument(
                content=content,
                document_id=doc_id,
                source_name=source_name,
                metadata={
                    "beir_doc_id": doc_id,
                    "title": title,
                    "source_name": source_name,
                },
            )
        )

    return selected_doc_ids, documents


def filter_queries_for_indexed_docs(
    queries: dict,
    qrels: dict,
    indexed_doc_ids: list[str],
    max_queries: int,
):
    indexed_doc_id_set = set(indexed_doc_ids)
    filtered_queries = {}
    filtered_qrels = {}

    for query_id, doc_relevances in qrels.items():
        supported_relevances = {
            doc_id: relevance
            for doc_id, relevance in doc_relevances.items()
            if doc_id in indexed_doc_id_set
        }
        if not supported_relevances:
            continue

        filtered_queries[query_id] = queries[query_id]
        filtered_qrels[query_id] = supported_relevances

        if len(filtered_queries) >= max_queries:
            break

    return filtered_queries, filtered_qrels


indexed_doc_ids, documents = build_source_documents(
    corpus=corpus,
    max_documents=MAX_CORPUS_DOCUMENTS,
)
evaluation_queries, evaluation_qrels = filter_queries_for_indexed_docs(
    queries=queries,
    qrels=qrels,
    indexed_doc_ids=indexed_doc_ids,
    max_queries=MAX_QUERIES,
)

len(documents), len(evaluation_queries)


(300, 21)

In [5]:
def index_documents_with_progress(manager, documents):
    reports = []
    for document in tqdm(
        documents,
        total=len(documents),
        desc="Chunking and indexing documents",
        unit="doc",
        dynamic_ncols=True,
    ):
        reports.append(manager.chunking.run([document]))

    return {
        "documents_processed": sum(
            report.documents_processed for report in reports
        ),
        "chunks_created": sum(report.chunks_created for report in reports),
        "collection_name": (
            reports[-1].collection_name if reports else QDRANT_COLLECTION_NAME
        ),
        "graph_namespace": (
            reports[-1].graph_namespace if reports else GRAPH_NAMESPACE
        ),
    }


print(
    "Preparing pipeline manager. On the first run this step may download Hugging Face models "
    "before the document indexing progress bar appears."
)

manager = PipelineManager(
    qdrant_url=QDRANT_URL,
    qdrant_collection_name=QDRANT_COLLECTION_NAME,
    neo4j_uri=NEO4J_URI,
    neo4j_username=NEO4J_USERNAME,
    neo4j_password=NEO4J_PASSWORD,
    llm_api_key=OPENROUTER_API_KEY,
    llm_model_name=OPENROUTER_MODEL_NAME,
    embedding_model_name=EMBEDDING_MODEL_NAME,
    reranker_model_name=RERANKER_MODEL_NAME,
    model_cache_dir=MODEL_CACHE_DIR,
    graph_namespace=GRAPH_NAMESPACE,
    neo4j_database=NEO4J_DATABASE,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    search_limit=SEARCH_LIMIT,
    search_candidate_limit=SEARCH_CANDIDATE_LIMIT,
)

print("Pipeline manager is ready. Starting document indexing.")

chunking_report = index_documents_with_progress(manager, documents)
chunking_report


Preparing pipeline manager. On the first run this step may download Hugging Face models before the document indexing progress bar appears.


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6662.58it/s]


Pipeline manager is ready. Starting document indexing.


Chunking and indexing documents:   7%|▋         | 21/300 [12:23<2:44:42, 35.42s/doc]


KeyboardInterrupt: 

In [ ]:
sample_query_id = next(iter(evaluation_queries))
sample_query = evaluation_queries[sample_query_id]
sample_report = manager.search.run(
    sample_query,
    limit=SEARCH_LIMIT,
    candidate_limit=SEARCH_CANDIDATE_LIMIT,
)

sample_rows = [
    {
        "chunk_id": item.chunk_id,
        "beir_doc_id": item.metadata.get("beir_doc_id", item.document_id),
        "title": item.metadata.get("title", ""),
        "final_score": item.final_score,
        "vector_score": item.vector_score,
        "graph_score": item.graph_score,
        "rerank_score": item.rerank_score,
    }
    for item in sample_report.items
]

pprint({
    "query_id": sample_query_id,
    "query": sample_query,
    "top_items": sample_rows,
})


In [ ]:
def aggregate_report_to_document_scores(search_report):
    document_scores = {}
    for item in search_report.items:
        doc_id = str(item.metadata.get("beir_doc_id") or item.document_id)
        current_score = document_scores.get(doc_id)
        if current_score is None or item.final_score > current_score:
            document_scores[doc_id] = item.final_score
    return document_scores


def build_results_for_queries(manager, queries: dict) -> dict:
    results = {}

    for query_id, query_text in tqdm(
        queries.items(),
        total=len(queries),
        desc="Running search evaluation",
        unit="query",
    ):
        report = manager.search.run(
            query_text,
            limit=SEARCH_LIMIT,
            candidate_limit=SEARCH_CANDIDATE_LIMIT,
        )
        results[query_id] = aggregate_report_to_document_scores(report)

    return results


search_results = build_results_for_queries(manager, evaluation_queries)
len(search_results)


In [ ]:
def ranked_doc_ids(result_scores: dict) -> list[str]:
    return [
        doc_id
        for doc_id, _ in sorted(
            result_scores.items(),
            key=lambda item: item[1],
            reverse=True,
        )
    ]


def dcg(relevances: list[float]) -> float:
    return sum(
        ((2 ** relevance) - 1) / math.log2(index + 2)
        for index, relevance in enumerate(relevances)
    )


def evaluate_retrieval_metrics(
    qrels: dict,
    results: dict,
    k_values: list[int],
) -> dict:
    metrics = {
        f"ndcg@{k}": 0.0 for k in k_values
    }
    metrics.update({f"recall@{k}": 0.0 for k in k_values})
    metrics.update({f"precision@{k}": 0.0 for k in k_values})
    metrics.update({f"mrr@{k}": 0.0 for k in k_values})

    query_count = len(qrels)
    for query_id, relevant_docs in qrels.items():
        ranked_ids = ranked_doc_ids(results.get(query_id, {}))
        relevant_doc_ids = {
            doc_id
            for doc_id, relevance in relevant_docs.items()
            if relevance > 0
        }
        ideal_relevances = sorted(relevant_docs.values(), reverse=True)

        for k in k_values:
            top_k_ids = ranked_ids[:k]
            top_k_relevances = [relevant_docs.get(doc_id, 0) for doc_id in top_k_ids]
            hits = sum(1 for doc_id in top_k_ids if doc_id in relevant_doc_ids)

            ndcg_key = f"ndcg@{k}"
            recall_key = f"recall@{k}"
            precision_key = f"precision@{k}"
            mrr_key = f"mrr@{k}"

            current_dcg = dcg(top_k_relevances)
            ideal_dcg = dcg(ideal_relevances[:k])
            metrics[ndcg_key] += current_dcg / ideal_dcg if ideal_dcg else 0.0
            metrics[recall_key] += hits / len(relevant_doc_ids) if relevant_doc_ids else 0.0
            metrics[precision_key] += hits / k

            reciprocal_rank = 0.0
            for rank_index, doc_id in enumerate(top_k_ids, start=1):
                if doc_id in relevant_doc_ids:
                    reciprocal_rank = 1.0 / rank_index
                    break
            metrics[mrr_key] += reciprocal_rank

    return {
        metric_name: round(metric_value / query_count, 4)
        for metric_name, metric_value in metrics.items()
    }


metrics = evaluate_retrieval_metrics(
    qrels=evaluation_qrels,
    results=search_results,
    k_values=K_VALUES,
)
pprint(metrics)


In [ ]:
# Close clients explicitly when the experiment is done.
manager.close()
